# 04 - Preprocesamiento

Pipeline de preprocesamiento que resuelve los tres problemas
identificados en el EDA: redundancia de features, distribuciones
asimétricas y diferencia de escala entre variables.

## 1. Carga de datos

In [22]:
import pandas as pd
import numpy as np
import supabase
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/asteroids_raw.csv')
print(f"Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")

Datos cargados: 22617 filas, 9 columnas


## 2. Feature Engineering

diameter_min_km y diameter_max_km presentaron correlación 1.00 en el EDA.
Mantener ambas agrega ruido sin información adicional. Se reemplazan
por diameter_mean_km = (min + max) / 2.

In [12]:
# Creamos el promedio entre min y max
df['diameter_mean_km'] = (df['diameter_min_km'] + df['diameter_max_km']) / 2

# Eliminamos las dos originales
df = df.drop(columns=['diameter_min_km', 'diameter_max_km'])

print("Columnas actuales:")
print(df.columns.tolist())

Columnas actuales:
['id', 'name', 'absolute_magnitude_h', 'velocity_km_per_hour', 'miss_distance_km', 'close_approach_date', 'is_potentially_hazardous', 'diameter_mean_km']


## 3. Transformación logarítmica

Según el EDA, diameter_mean_km, velocity_km_per_hour y miss_distance_km presentan
distribuciones con cola derecha pronunciada. Se aplica np.log1p(), equivalente a log(x+1), para estabilizar la varianza y reducir
el impacto de outliers. Se usa log1p en lugar de log para manejar
correctamente valores cercanos a cero.

absolute_magnitude_h no requiere transformación, distribución
aproximadamente normal confirmada en EDA.

In [13]:
features_a_transformar = ['diameter_mean_km', 'velocity_km_per_hour', 'miss_distance_km']

for feature in features_a_transformar:
    df[f'{feature}_log'] = np.log1p(df[feature])
    df = df.drop(columns=[feature])

print("Columnas después de transformación logarítmica:")
print(df.columns.tolist())

Columnas después de transformación logarítmica:
['id', 'name', 'absolute_magnitude_h', 'close_approach_date', 'is_potentially_hazardous', 'diameter_mean_km_log', 'velocity_km_per_hour_log', 'miss_distance_km_log']


## 4. Separación X / y

Se excluyen id, name y close_approach_date por ser metadata sin
poder predictivo. El dataset resultante tiene 4 features numéricas
y 1 target binario.

In [14]:
X = df.drop(columns=['id', 'name', 'close_approach_date', 'is_potentially_hazardous'])
y = df['is_potentially_hazardous']

print("Features (X):")
print(X.columns.tolist())
print(f"\nDimensiones de X: {X.shape}")
print(f"Dimensiones de y: {y.shape}")
print(f"\nDistribución del target:")
print(y.value_counts())

Features (X):
['absolute_magnitude_h', 'diameter_mean_km_log', 'velocity_km_per_hour_log', 'miss_distance_km_log']

Dimensiones de X: (22617, 4)
Dimensiones de y: (22617,)

Distribución del target:
is_potentially_hazardous
False    21357
True      1260
Name: count, dtype: int64


## 5. Split train/test

Split 80/20 con stratify=y. La estratificación garantiza que la
proporción 94.4/5.6 se preserve en ambos sets. Sin esto, la
distribución de clases podría diferir entre train y test por azar,
invalidando la evaluación.

El test set no se toca hasta la evaluación final de modelos.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y # ayuda a que quede la misma proporción en ambos sets
)

print(f"Train: {X_train.shape[0]} filas")
print(f"Test:  {X_test.shape[0]} filas")
print(f"\nDistribución en train:")
print(y_train.value_counts())
print(f"\nDistribución en test:")
print(y_test.value_counts())

Train: 18093 filas
Test:  4524 filas

Distribución en train:
is_potentially_hazardous
False    17085
True      1008
Name: count, dtype: int64

Distribución en test:
is_potentially_hazardous
False    4272
True      252
Name: count, dtype: int64


In [18]:
print("Proporción en train:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nProporción en test:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Proporción en train:
is_potentially_hazardous
False    94.43
True      5.57
Name: proportion, dtype: float64

Proporción en test:
is_potentially_hazardous
False    94.43
True      5.57
Name: proportion, dtype: float64


## 6. Escalado de features

StandardScaler aplicado con fit solo sobre X_train para evitar
data leakage. Si se entrenara sobre el dataset completo, el modelo
accedería indirectamente a información del test set durante el
preprocesamiento.

In [20]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Media de cada feature en train (debe ser ~0):")
print(X_train_scaled.mean().round(4))
print("\nDesvío estándar de cada feature en train (debe ser ~1):")
print(X_train_scaled.std().round(4))

Media de cada feature en train (debe ser ~0):
absolute_magnitude_h       -0.0
diameter_mean_km_log        0.0
velocity_km_per_hour_log   -0.0
miss_distance_km_log        0.0
dtype: float64

Desvío estándar de cada feature en train (debe ser ~1):
absolute_magnitude_h        1.0
diameter_mean_km_log        1.0
velocity_km_per_hour_log    1.0
miss_distance_km_log        1.0
dtype: float64


Resultado: media ~0 y desvío estándar ~1 en todas las features
del train set.

## 7. Guardado de datos procesados y scaler

In [21]:
X_train_scaled.to_csv('../data/X_train.csv', index=False)
X_test_scaled.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

joblib.dump(scaler, '../data/scaler.pkl')

print("Archivos guardados en /data:")
print("  ✓ X_train.csv")
print("  ✓ X_test.csv")
print("  ✓ y_train.csv")
print("  ✓ y_test.csv")
print("  ✓ scaler.pkl")

Archivos guardados en /data:
  ✓ X_train.csv
  ✓ X_test.csv
  ✓ y_train.csv
  ✓ y_test.csv
  ✓ scaler.pkl


## 8. Feature Store

Las features procesadas se persisten en la tabla asteroids_features
de Supabase, separando los datos raw (asteroids) de las features
listas para ML. Cualquier notebook del proyecto consume esta tabla
directamente sin necesidad de reprocesar.

Los splits y el scaler se serializan localmente en /data para
reproducibilidad. Ver README para instrucciones de setup.

In [23]:
supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

# Combinamos X e y para insertar juntos
df_features = X.copy()
df_features['is_potentially_hazardous'] = y.values
df_features['id'] = df['id'].values

print("Cargando en asteroids_features...")
batch_size = 500
for i in range(0, len(df_features), batch_size):
    batch = df_features.iloc[i:i+batch_size].to_dict(orient='records')
    supabase.table('asteroids_features').upsert(batch).execute()
    print(f"  ✓ Insertados registros {i} a {i+len(batch)}")

print(f"\nListo. {len(df_features)} registros en asteroids_features")

Cargando en asteroids_features...
  ✓ Insertados registros 0 a 500
  ✓ Insertados registros 500 a 1000
  ✓ Insertados registros 1000 a 1500
  ✓ Insertados registros 1500 a 2000
  ✓ Insertados registros 2000 a 2500
  ✓ Insertados registros 2500 a 3000
  ✓ Insertados registros 3000 a 3500
  ✓ Insertados registros 3500 a 4000
  ✓ Insertados registros 4000 a 4500
  ✓ Insertados registros 4500 a 5000
  ✓ Insertados registros 5000 a 5500
  ✓ Insertados registros 5500 a 6000
  ✓ Insertados registros 6000 a 6500
  ✓ Insertados registros 6500 a 7000
  ✓ Insertados registros 7000 a 7500
  ✓ Insertados registros 7500 a 8000
  ✓ Insertados registros 8000 a 8500
  ✓ Insertados registros 8500 a 9000
  ✓ Insertados registros 9000 a 9500
  ✓ Insertados registros 9500 a 10000
  ✓ Insertados registros 10000 a 10500
  ✓ Insertados registros 10500 a 11000
  ✓ Insertados registros 11000 a 11500
  ✓ Insertados registros 11500 a 12000
  ✓ Insertados registros 12000 a 12500
  ✓ Insertados registros 12500 a 13